### We are using the leNet5 CNN for this


In [31]:
from pathlib import Path
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from scipy.ndimage import center_of_mass, shift



In [32]:
TrainingDirectory = Path("iivp-2026-challenge/train/train")
X_train, y_train = [], []
for class_dir in sorted(TrainingDirectory.iterdir(), key=lambda p: int(p.name)):
    label = int(class_dir.name)
    for png in class_dir.glob("*.png"):
        X_train.append(np.array(Image.open(png), dtype=np.uint8))
        y_train.append(label)

X_train = np.stack(X_train)
y_train = np.array(y_train)

print(X_train.shape, y_train.shape)

def recenter(img):
    cy,cx = center_of_mass(img)
    if np.isnan(cy):
        return img
    return shift(img, (img.shape[0]/2 - cy, img.shape[1]/2 -cx), order=1)

X_train_centered = np.array([recenter(img) for img in X_train])


(17000, 32, 32) (17000,)


In [33]:
class LeNet5(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1,6, kernel_size=5)
        self.pool = nn.MaxPool2d(2,2)
        self.conv2 = nn.Conv2d(6,16,kernel_size=5)
        self.fc1 = nn.Linear(16*5*5, 120)
        self.fc2 = nn.Linear(120,84)
        self.fc3 = nn.Linear(84, num_classes)


    def forward(self,x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x,1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

In [34]:
model = LeNet5()
dummy = torch.randn(1,1,32,32)
print(model(dummy).shape)

torch.Size([1, 10])


In [35]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_centered, y_train,
    test_size=0.2, stratify=y_train, random_state=42,
)

X_tr_t  = torch.tensor(X_tr,  dtype=torch.float32).unsqueeze(1) / 255.0
X_val_t = torch.tensor(X_val, dtype=torch.float32).unsqueeze(1) / 255.0
y_tr_t  = torch.tensor(y_tr,  dtype=torch.long)
y_val_t = torch.tensor(y_val, dtype=torch.long)

train_ds = TensorDataset(X_tr_t, y_tr_t)
val_ds   = TensorDataset(X_val_t, y_val_t)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=256, shuffle=False)

print(X_tr_t.shape, y_tr_t.shape)


torch.Size([13600, 1, 32, 32]) torch.Size([13600])


In [36]:
dev = "mps"
model = LeNet5().to(dev)
optimizer = torch.optim.Adam(model.parameters(),lr=1e-3)
criterion = nn.CrossEntropyLoss()

In [ ]:
def evaluate(loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xd, yd in loader:
            xd, yd = xd.to(dev), yd.to(dev)
            pred = model(xd).argmax(dim=1)
            correct += (pred == yd).sum().item()
            total += yd.size(0)
    return correct / total

epochs = 10
for epoch in range(1, epochs + 1):
    model.train()
    run_loss = 0.0
    for xd, yd in train_loader:
        xd, yd = xd.to(dev), yd.to(dev)
        optimizer.zero_grad()
        loss = criterion(model(xd),yd)
        loss.backward()
        optimizer.step()
        run_loss += loss.item() * xd.size(0)

    train_loss = run_loss / len(train_ds)
    val_acc = evaluate(val_loader)
    print(f"epoch {epoch:2d}  loss {train_loss:.4f}  val_acc {val_acc:.4f}")

    

NameError: name 'running_loss' is not defined